# 07b: BERT on Colab (A4, A5)

Untrained and fine-tuned `bert-base-uncased` on the jev benchmarks MCQ data, evaluated on the validation set and test Levels 0-3.

**Before running**
1. On a laptop: `python 07a_build_bert_data.py`, then copy `01-jev-benchmarks/data/bert/` to Google Drive as `MyDrive/jev-bert/data/`.
2. Here: *Runtime → Change runtime type → T4 GPU* (or *v5e-1 TPU* if no GPU is free), then *Run all*.

On a GPU it trains in fp16; on a TPU (via `torch_xla`) in bf16, so the two give slightly different numbers. Use one accelerator for all seeds. `train_log.json` records which one (`gpu`, `settings.precision`). On a TPU the first batches of each new shape are slow while XLA compiles, which inflates their `latency_s`.

**What it does**
- **Untrained** (chance floor): pre-trained encoder with a randomly initialised multiple-choice head, no fine-tuning, seed `SEEDS[0]`.
- **Fine-tuned**, one model per seed: trained on `train.jsonl` (originals + paraphrases), best epoch picked on `val.jsonl` accuracy.
- Each model predicts `val` and every `level<n>.jsonl` present.
- **Learning curve** (diagnostic, `LEARNING_CURVE = True`; not the main run): the same training on nested subsets of the train questions (`CURVE_FRACTIONS` of each dataset, a question's original and paraphrase kept together, the same subset for every seed), written to `results_learning_curve/frac_<pct>_seed_<n>/`. Shows per dataset whether more training data still raises accuracy. About 80 min on a T4 with the defaults.

Options are padded to 5 with empty dummies whose logits are masked to −∞ (A2 rule 6), so 3-, 4- and 5-option questions share one model.

**Output** in `MyDrive/jev-bert/results/<untrained | seed_<n>>/`: `<run>.jsonl` (same record format as the Jev and LLM results, see `helpers/evaluation.py`) and `train_log.json`. A finished seed is skipped on re-run, so a Colab disconnect only loses the seed in progress. Download `MyDrive/jev-bert/results/` into `01-jev-benchmarks/results/bert/` and commit it.

In [ ]:
# Settings
MODEL_NAME = "bert-base-uncased"
SEEDS = (11, 22, 33, 44, 55)  # = BERT_SEEDS in helpers/config.py
RUN_UNTRAINED = True
TRAIN_VERSIONS = ("original", "paraphrase")  # A5 main run; one of them alone = the optional ablations
RESULTS_NAME = "results"  # e.g. "results_original_only" for an ablation, so the main run is not overwritten

# First Colab run: 1 seed, SMOKE_TRAIN_ROWS random train rows, 1 epoch, written to results_smoke/
# (never mixed with real results). Check the truncation count, that the loss falls and val is above chance.
SMOKE_TEST = False
SMOKE_TRAIN_ROWS = 2_000

MAX_OPTIONS = 5
MAX_LENGTH = 192  # longest question + option is ~126 words; truncations are counted below
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1

# Learning curve: train on nested subsets of the train questions instead of the main run (see the top).
# Fractions are of each dataset's questions; 1.0 repeats the main run's data with CURVE_SEEDS.
LEARNING_CURVE = False
CURVE_FRACTIONS = (0.25, 0.5, 1.0)
CURVE_SEEDS = SEEDS[:2]

if SMOKE_TEST:
    SEEDS, EPOCHS, RUN_UNTRAINED, RESULTS_NAME = SEEDS[:1], 1, False, "results_smoke"
    CURVE_SEEDS = SEEDS
if LEARNING_CURVE:
    RUN_UNTRAINED = False
    RESULTS_NAME = "results_smoke_curve" if SMOKE_TEST else "results_learning_curve"

In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    # Kaggle: data/bert/ uploaded as a dataset (any name); results go to the notebook's output
    DATA_DIR = next(KAGGLE_INPUT.rglob("train.jsonl")).parent
    RESULTS_DIR = Path("/kaggle/working") / RESULTS_NAME
else:
    from google.colab import drive

    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/jev-bert")
    DATA_DIR = ROOT / "data"
    RESULTS_DIR = ROOT / RESULTS_NAME
print(DATA_DIR, sorted(p.name for p in DATA_DIR.glob("*")))

In [ ]:
import json
import random
import time

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForMultipleChoice, AutoTokenizer, get_linear_schedule_with_warmup

# CUDA GPU (fp16 + GradScaler) if there is one, else a Colab TPU via torch_xla (bf16, no scaler needed)
if torch.cuda.is_available():
    BACKEND, DEVICE, AMP_DTYPE = "cuda", torch.device("cuda"), torch.float16
    GPU_NAME = torch.cuda.get_device_name(0)
else:
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
    except ImportError:
        raise RuntimeError("No GPU or TPU: Runtime -> Change runtime type -> T4 GPU or v5e-1 TPU")
    BACKEND, DEVICE, AMP_DTYPE = "xla", torch_xla.device(), torch.bfloat16
    GPU_NAME = f"{xm.xla_device_hw(DEVICE)} (torch_xla {torch_xla.__version__})"
print(GPU_NAME, "| torch", torch.__version__)
LETTERS = "ABCDE"


def sync():
    """Wait for queued device work, so wall-clock timings are real."""
    if BACKEND == "cuda":
        torch.cuda.synchronize()
    else:
        xm.wait_device_ops()

## Data

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


train_rows = [r for r in read_jsonl(DATA_DIR / "train.jsonl") if r["version"] in TRAIN_VERSIONS]
if SMOKE_TEST:
    train_rows = random.Random(0).sample(train_rows, min(SMOKE_TRAIN_ROWS, len(train_rows)))
eval_sets = {"val": read_jsonl(DATA_DIR / "val.jsonl")}
for level in range(4):
    path = DATA_DIR / f"level{level}.jsonl"
    if path.exists():
        eval_sets[f"level{level}"] = read_jsonl(path)
    else:
        print(f"level{level}: not in {DATA_DIR}, skipped")
print("train", len(train_rows), {name: len(rows) for name, rows in eval_sets.items()})


def curve_subset(fraction):
    """Indices into train_rows: the first `fraction` of each dataset's questions in a fixed random order.

    Nested (a smaller fraction is a subset of a larger one), the same for every seed, and a question's
    original and paraphrase are kept or dropped together.
    """
    rows_by_question = {}
    for i, row in enumerate(train_rows):
        rows_by_question.setdefault(row["dataset"], {}).setdefault(row["id"], []).append(i)
    index = []
    for dataset, questions in sorted(rows_by_question.items()):
        ids = sorted(questions)
        random.Random(f"curve:{dataset}").shuffle(ids)
        for question_id in ids[: round(fraction * len(ids))]:
            index += questions[question_id]
    return torch.tensor(sorted(index))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(rows):
    """Tensors of shape (questions, MAX_OPTIONS, MAX_LENGTH) plus an option mask and labels.

    Each option is the pair (question, option); dummy options are empty and masked.
    """
    firsts, seconds, mask = [], [], []
    for row in rows:
        options = row["options"] + [""] * (MAX_OPTIONS - len(row["options"]))
        firsts += [row["question"]] * MAX_OPTIONS
        seconds += options
        mask.append([i < len(row["options"]) for i in range(MAX_OPTIONS)])
    untruncated = tokenizer(firsts, seconds, truncation=False)["input_ids"]
    too_long = sum(len(ids) > MAX_LENGTH for ids in untruncated)
    encoded = tokenizer(firsts, seconds, truncation="longest_first", max_length=MAX_LENGTH,
                        padding="max_length", return_tensors="pt")
    shape = (len(rows), MAX_OPTIONS, MAX_LENGTH)
    return {
        "input_ids": encoded["input_ids"].view(shape),
        "attention_mask": encoded["attention_mask"].view(shape),
        "token_type_ids": encoded["token_type_ids"].view(shape),
        "option_mask": torch.tensor(mask),
        "labels": torch.tensor([row["label"] for row in rows]),
        "truncated": too_long,
    }


train_data = encode(train_rows)
eval_data = {name: encode(rows) for name, rows in eval_sets.items()}
print("truncated (question, option) pairs:", train_data["truncated"], {n: d["truncated"] for n, d in eval_data.items()})

## Model, training, prediction

In [ ]:
TENSOR_KEYS = ("input_ids", "attention_mask", "token_type_ids")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if BACKEND == "cuda":
        torch.cuda.manual_seed_all(seed)
    else:
        xm.set_rng_state(seed)  # dropout on the TPU uses its own RNG


def masked_logits(model, batch):
    """Logits per option, with dummy options at -inf (in float32, so fp16/bf16 can't overflow)."""
    with torch.autocast(BACKEND, dtype=AMP_DTYPE):
        logits = model(**{k: batch[k] for k in TENSOR_KEYS}).logits
    return logits.float().masked_fill(~batch["option_mask"], float("-inf"))


def batches(data, size, order):
    for start in range(0, len(order), size):
        index = order[start:start + size]
        yield {k: data[k][index].to(DEVICE) for k in (*TENSOR_KEYS, "option_mask", "labels")}


@torch.no_grad()
def predict(model, name):
    """Records in the jev benchmarks result format (see helpers/evaluation.py)."""
    model.eval()
    data, rows = eval_data[name], eval_sets[name]
    records, order = [], torch.arange(len(rows))
    for start, batch in zip(range(0, len(rows), EVAL_BATCH_SIZE), batches(data, EVAL_BATCH_SIZE, order)):
        sync()
        began = time.perf_counter()
        probabilities = masked_logits(model, batch).softmax(dim=-1).cpu()
        sync()
        per_question = (time.perf_counter() - began) / len(probabilities)
        for row, probs in zip(rows[start:start + EVAL_BATCH_SIZE], probabilities):
            n = len(row["options"])
            pred = LETTERS[int(probs[:n].argmax())]
            records.append({
                "id": row["id"], "dataset": row["dataset"], "run": name, "quiz_set": None,
                "n_options": n, "answer": row["answer"], "pred": pred,
                "probabilities": {LETTERS[i]: float(probs[i]) for i in range(n)},
                "correct": pred == row["answer"], "latency_s": round(per_question, 6),
                "cost_usd": 0.0, "cached": False, "error": None,
                "version": row["version"],
            })
    return records


def accuracy(records):
    return sum(r["correct"] for r in records) / len(records)


def train(seed, index=None):
    """Fine-tune one model on train_rows[index] (default: all); keep the epoch with the best val accuracy.

    Returns (model, log).
    """
    index = torch.arange(len(train_rows)) if index is None else index
    set_seed(seed)
    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
    steps = EPOCHS * -(-len(index) // BATCH_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=BACKEND == "cuda")  # bf16 needs no loss scaling
    generator = torch.Generator().manual_seed(seed)
    log, best_acc, best_state = {"epochs": []}, -1.0, None
    began = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = index[torch.randperm(len(index), generator=generator)]
        losses = []
        for batch in batches(train_data, BATCH_SIZE, order):
            loss = F.cross_entropy(masked_logits(model, batch), batch["labels"])
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if BACKEND == "xla":
                xm.mark_step()  # run this step's graph on the TPU
            losses.append(loss.detach())  # no .item() per step: it would stall the TPU
        mean_loss = torch.stack(losses).mean().item()
        val_acc = accuracy(predict(model, "val"))
        log["epochs"].append({"epoch": epoch, "train_loss": mean_loss, "val_accuracy": val_acc})
        print(f"  seed {seed} epoch {epoch}: loss {mean_loss:.4f}, val {val_acc:.1%}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    log |= {
        "seed": seed, "best_epoch": max(log["epochs"], key=lambda e: e["val_accuracy"])["epoch"],
        "train_seconds": round(time.perf_counter() - began, 1), "gpu": GPU_NAME,
        "train_rows": len(index), "train_versions": list(TRAIN_VERSIONS),
        "settings": {"model": MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
                     "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup": WARMUP_FRACTION,
                     "precision": str(AMP_DTYPE).removeprefix("torch.")},
    }
    return model, log


def save(folder, model, log):
    """Predict every eval set and write <run>.jsonl + train_log.json (last, marking the folder done)."""
    folder.mkdir(parents=True, exist_ok=True)
    summary = {}
    for name in eval_sets:
        records = predict(model, name)
        with open(folder / f"{name}.jsonl", "w", encoding="utf-8") as handle:
            handle.writelines(json.dumps(r) + "\n" for r in records)
        summary[name] = accuracy(records)
    log["accuracy"] = summary
    (folder / "train_log.json").write_text(json.dumps(log, indent=1))
    print(" ", folder.name, {k: f"{v:.1%}" for k, v in summary.items()})

## Run

In [ ]:
def done(folder):
    return (folder / "train_log.json").exists()


if RUN_UNTRAINED and not done(RESULTS_DIR / "untrained"):
    set_seed(SEEDS[0])
    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
    save(RESULTS_DIR / "untrained", model, {"seed": SEEDS[0], "trained": False, "gpu": GPU_NAME})
    del model

if LEARNING_CURVE:
    jobs = [(RESULTS_DIR / f"frac_{round(100 * fraction):03d}_seed_{seed}", seed, fraction)
            for fraction in CURVE_FRACTIONS for seed in CURVE_SEEDS]
else:
    jobs = [(RESULTS_DIR / f"seed_{seed}", seed, None) for seed in SEEDS]

for folder, seed, fraction in jobs:
    if done(folder):
        print(f"{folder.name}: already done, skipped")
        continue
    index = None if fraction is None else curve_subset(fraction)
    model, log = train(seed, index)
    if fraction is not None:
        log["curve_fraction"] = fraction
    save(folder, model, log)
    del model
    if BACKEND == "cuda":
        torch.cuda.empty_cache()

## Summary

Accuracy per dataset (mean ± SD over seeds). The above-chance rule and all statistics are computed by `10_analyse.py`.

Learning curve: mean accuracy over `CURVE_SEEDS` per training fraction, on val originals (200 per dataset) and Level 0 (400 per dataset). One fraction's val number has a 95% CI of about ±7 points per dataset, so read the trend across fractions, not single steps.

In [ ]:
import pandas as pd

rows = []
for folder in sorted(RESULTS_DIR.glob("*")):
    for path in folder.glob("*.jsonl"):
        for r in read_jsonl(path):
            rows.append({"model": folder.name, "run": path.stem, "dataset": r["dataset"],
                         "version": r["version"], "correct": r["correct"]})
results = pd.DataFrame(rows)

if LEARNING_CURVE:
    curve = results[(results["run"] == "level0") | ((results["run"] == "val") & (results["version"] == "original"))].copy()
    curve["fraction"] = curve["model"].str.extract(r"frac_(\d+)_")[0].astype(int) / 100
    per_model = curve.groupby(["run", "fraction", "model", "dataset"])["correct"].mean()
    print(per_model.groupby(level=["run", "fraction", "dataset"]).mean().unstack("dataset").round(3).to_string())
else:
    table = results.groupby(["model", "run", "dataset"])["correct"].mean().unstack("dataset")
    seeds = table.drop(index="untrained", level="model", errors="ignore")
    print(seeds.groupby(level="run").agg(["mean", "std"]).round(3).to_string())
    print(table.loc["untrained"].round(3).to_string() if "untrained" in table.index.get_level_values(0) else "")